In [1]:
!pip install langchain langchain-core langchain-groq langgraph groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.3 MB/s eta 0:00:00


In [2]:
import os
os.environ["GROQ_API_KEY"] = "your groq_api_key here"

In [3]:
from google.colab import files
uploaded = files.upload()


Saving knowledge_base.json to knowledge_base.json


In [4]:
import json
import os
import re
from typing import Annotated, TypedDict, Optional
from pathlib import Path

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

# Load knowledge base (Colab-compatible path)
with open("knowledge_base.json", "r") as f:
    kb = json.load(f)

KNOWLEDGE_BASE = f"""
COMPANY: {kb['company']}
DESCRIPTION: {kb['description']}

PRICING PLANS:
- Basic Plan: {kb['pricing']['basic_plan']['price']}
  Features: {', '.join(kb['pricing']['basic_plan']['features'])}

- Pro Plan: {kb['pricing']['pro_plan']['price']}
  Features: {', '.join(kb['pricing']['pro_plan']['features'])}

COMPANY POLICIES:
- Refund Policy: {kb['policies']['refund_policy']}
- Support Policy: {kb['policies']['support_policy']}
- Free Trial: {kb['policies']['free_trial']}
- Cancellation: {kb['policies']['cancellation']}

SUPPORTED PLATFORMS: {', '.join(kb['features_overview']['supported_platforms'])}
"""

In [7]:
def mock_lead_capture(name, email, platform):
    print(f"\n✅ Lead captured successfully: {name}, {email}, {platform}\n")
    return f"Lead captured successfully: {name}, {email}, {platform}"

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    intent: str
    lead_name: Optional[str]
    lead_email: Optional[str]
    lead_platform: Optional[str]
    lead_captured: bool
    awaiting_field: Optional[str]



def get_llm():
    return ChatGroq(model="llama-3.1-8b-instant", temperature=0.3,api_key="your groq_api_key here")

INTENT_SYSTEM_PROMPT = """You are an intent classifier for AutoStream, a SaaS video editing platform.
Classify the user's latest message into exactly ONE of these intents:
- "greeting"
- "product_inquiry"
- "high_intent"
Respond with ONLY the intent label, nothing else."""

AGENT_SYSTEM_PROMPT = """You are a friendly sales assistant for AutoStream — an AI-powered video editing SaaS.
Use ONLY the knowledge base below to answer questions. Do not make up information.

--- KNOWLEDGE BASE ---
""" + KNOWLEDGE_BASE + """
--- END KNOWLEDGE BASE ---

Be conversational, warm, and helpful. Keep responses concise (2-4 sentences)."""

def detect_intent(state):
    llm = get_llm()
    last_human = next((m.content for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), "").lower()

    # Keyword-based detection first (more reliable)
    high_intent_keywords = ["sign up", "signup", "subscribe", "buy", "purchase", "get started",
                             "try", "want to", "ready to", "start", "join", "pro plan", "basic plan"]
    product_keywords = ["price", "pricing", "plan", "cost", "feature", "include", "offer",
                        "what is", "how much", "refund", "support", "resolution", "caption"]

    if any(word in last_human for word in high_intent_keywords):
        intent = "high_intent"
    elif any(word in last_human for word in product_keywords):
        intent = "product_inquiry"
    else:
        # Fall back to LLM only if keywords don't match
        response = llm.invoke([SystemMessage(content=INTENT_SYSTEM_PROMPT), HumanMessage(content=last_human)])
        raw = response.content.strip().lower()
        if "high_intent" in raw:
            intent = "high_intent"
        elif "product" in raw or "inquiry" in raw:
            intent = "product_inquiry"
        else:
            intent = "greeting"

    return {**state, "intent": intent}

def collect_lead_info(state):
    last_human = next((m.content for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), "").strip()
    name, email, platform = state.get("lead_name"), state.get("lead_email"), state.get("lead_platform")
    awaiting = state.get("awaiting_field")

    if awaiting == "name" and not name:
        name = last_human
    elif awaiting == "email" and not email:
        if re.match(r"[^@]+@[^@]+\.[^@]+", last_human):
            email = last_human
        else:
            ai_msg = AIMessage(content="That doesn't look like a valid email. Could you re-enter it?")
            return {**state, "messages": state["messages"] + [ai_msg], "awaiting_field": "email"}
    elif awaiting == "platform" and not platform:
        platform = last_human

    if not name:
        ai_msg = AIMessage(content="Great choice! I'd love to get you set up. Could I start with your name?")
        return {**state, "messages": state["messages"] + [ai_msg], "lead_name": name, "lead_email": email, "lead_platform": platform, "awaiting_field": "name"}
    if not email:
        ai_msg = AIMessage(content=f"Nice to meet you, {name}! What's your email address?")
        return {**state, "messages": state["messages"] + [ai_msg], "lead_name": name, "lead_email": email, "lead_platform": platform, "awaiting_field": "email"}
    if not platform:
        ai_msg = AIMessage(content="Almost there! Which creator platform do you mainly publish on? (e.g. YouTube, Instagram, TikTok)")
        return {**state, "messages": state["messages"] + [ai_msg], "lead_name": name, "lead_email": email, "lead_platform": platform, "awaiting_field": "platform"}

    mock_lead_capture(name, email, platform)
    ai_msg = AIMessage(content=f"🎉 You're all set, {name}! We've captured your details and our team will reach out shortly. Welcome to AutoStream!")
    return {**state, "messages": state["messages"] + [ai_msg], "lead_name": name, "lead_email": email, "lead_platform": platform, "lead_captured": True, "awaiting_field": None}

def generate_response(state):
    llm = get_llm()
    intent = state.get("intent", "greeting")
    if state.get("awaiting_field") or intent == "high_intent":
        if not state.get("lead_captured"):
            return collect_lead_info(state)
    system_msg = SystemMessage(content=AGENT_SYSTEM_PROMPT)
    response = llm.invoke([system_msg] + state["messages"])
    return {**state, "messages": state["messages"] + [AIMessage(content=response.content)]}

def build_graph():
    graph = StateGraph(AgentState)
    graph.add_node("detect_intent", detect_intent)
    graph.add_node("respond", generate_response)
    graph.set_entry_point("detect_intent")
    graph.add_conditional_edges("detect_intent", lambda s: "respond", {"respond": "respond"})
    graph.add_edge("respond", END)
    return graph.compile()

print("✅ Agent ready! Run the next cell to start chatting.")

✅ Agent ready! Run the next cell to start chatting.


In [ ]:
app = build_graph()
state = {"messages": [], "intent": "greeting", "lead_name": None, "lead_email": None, "lead_platform": None, "lead_captured": False, "awaiting_field": None}

while True:
    user_input = input("\nYou: ").strip()
    if user_input.lower() in ("quit", "exit"):
        print("Agent: Goodbye! 👋")
        break
    state["messages"] = state["messages"] + [HumanMessage(content=user_input)]
    state = app.invoke(state)
    last_ai = next((m.content for m in reversed(state["messages"]) if isinstance(m, AIMessage)), "")
    print(f"\nAgent: {last_ai}")


You: Hi there!

Agent: Welcome to AutoStream! I'm here to help you with any questions or concerns about our AI-powered video editing tools. How can I assist you today?

You: i wanted to know about the plans and prices

Agent: We have two main plans: the Basic Plan for $29/month and the Pro Plan for $79/month. The Basic Plan includes 10 videos per month, 720p resolution, and basic auto-editing tools, with email support. The Pro Plan offers unlimited videos, 4K resolution, AI captions, advanced auto-editing tools, and 24/7 priority support.

You: i wil go for pro plan

Agent: Great choice! I'd love to get you set up. Could I start with your name?

You: Aman

Agent: Nice to meet you, Aman! What's your email address?

You: aman@gmail.com

Agent: Almost there! Which creator platform do you mainly publish on? (e.g. YouTube, Instagram, TikTok)

You: youtube

✅ Lead captured successfully: Aman, aman@gmail.com, youtube


Agent: 🎉 You're all set, Aman! We've captured your details and our team w